# National Job Market Predictor

This notebook uses BLS/OEWS national occupation files from 2010-2024 to build an occupation-level forecasting model. Wage modeling uses annual wage fields only; hourly wage columns are removed before feature engineering.

**Model choice:** Random Forest regression is used as the main model. It predicts each occupation's next-year employment and annual mean wage from the current year's occupation, employment, annual wage, and SOC-code features.

## Report Summary

### Data Used

The notebook reads national BLS/OEWS occupation-level CSV files from both `converted_csv/` and the original `oesm*nat/` folders. The available national files cover 2010-2024.

### Preprocessing

- Standardizes schema changes such as `GROUP`, `OCC_GROUP`, and `O_GROUP`.
- Extracts the year from each file name.
- Converts BLS numeric strings such as `"127,097,160"` into numbers.
- Treats suppression markers such as `*`, `**`, `#`, and blanks as missing values.
- Drops hourly wage fields and keeps annual wage fields only.
- Keeps detailed occupation rows and removes aggregate total/major/minor/broad rows.
- Builds a panel dataset with one row per `year + occupation`.
- Creates lagged one-year-ahead targets for employment and annual mean wage.

### Modeling

The model predicts next-year `TOT_EMP` and next-year annual mean wage `A_MEAN`. It uses a `RandomForestRegressor`, which averages many decision trees so the final prediction is less dependent on any single tree. The trees learn nonlinear relationships between prior-year employment, annual wage levels, wage percentiles, SOC-code groupings, occupation title/code, and year.

### Validation

The notebook uses a time-aware split. It trains on transitions whose target year is 2022 or earlier, then tests on 2023 and 2024 target years. This avoids training directly on the outcomes being scored.

In [1]:
from pathlib import Path
import re

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd()
CONVERTED_ROOT = PROJECT_ROOT / "converted_csv"

In [2]:
def extract_year(path: Path) -> int:
    match = re.search(r"M(20\d{2})", path.name)
    if not match:
        match = re.search(r"oesm(\d{2})", str(path))
    if not match:
        raise ValueError(f"Could not determine year from {path}")
    year_text = match.group(1)
    return int(year_text if len(year_text) == 4 else f"20{year_text}")


def discover_national_files(project_root: Path, start_year: int = 2010, end_year: int = 2024) -> list[Path]:
    converted = list((project_root / "converted_csv").glob("oesm*nat/national_M*_dl__national*.csv"))
    original = list(project_root.glob("oesm*nat/national_M*_dl.csv"))

    by_year: dict[int, Path] = {}
    for path in sorted(converted + original):
        year = extract_year(path)
        if not start_year <= year <= end_year:
            continue
        # Prefer converted files, but fall back to original CSVs where needed.
        if year not in by_year or "converted_csv" in path.parts:
            by_year[year] = path
    return [by_year[year] for year in sorted(by_year)]


national_files = discover_national_files(PROJECT_ROOT)
print(f"Found {len(national_files)} national occupation files")
for path in national_files:
    print(extract_year(path), path.relative_to(PROJECT_ROOT))

Found 15 national occupation files
2010 converted_csv/oesm10nat/national_M2010_dl__national_dl.csv
2011 oesm11nat/national_M2011_dl.csv
2012 converted_csv/oesm12nat/national_M2012_dl__national_dl.csv
2013 converted_csv/oesm13nat/national_M2013_dl__national_dl.csv
2014 converted_csv/oesm14nat/national_M2014_dl__national_dl.csv
2015 converted_csv/oesm15nat/national_M2015_dl__national_dl.csv
2016 converted_csv/oesm16nat/national_M2016_dl__national_dl.csv
2017 converted_csv/oesm17nat/national_M2017_dl__national_dl.csv
2018 converted_csv/oesm18nat/national_M2018_dl__national_dl.csv
2019 converted_csv/oesm19nat/national_M2019_dl__national_M2019_dl.csv
2020 converted_csv/oesm20nat/national_M2020_dl__national_M2020_dl.csv
2021 converted_csv/oesm21nat/national_M2021_dl__national_M2021_dl.csv
2022 converted_csv/oesm22nat/national_M2022_dl__national_M2022_dl.csv
2023 converted_csv/oesm23nat/national_M2023_dl__national_M2023_dl.csv
2024 converted_csv/oesm24nat/national_M2024_dl__national_M2024_dl.

In [3]:
SUPPRESSED_VALUES = {"", "*", "**", "#", "~"}
HOURLY_COLUMNS = {"H_MEAN", "H_PCT10", "H_PCT25", "H_MEDIAN", "H_PCT75", "H_PCT90", "HOURLY"}
NUMERIC_COLUMNS = [
    "TOT_EMP",
    "EMP_PRSE",
    "A_MEAN",
    "MEAN_PRSE",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90",
]


def clean_numeric(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .replace(list(SUPPRESSED_VALUES), pd.NA)
    )
    return pd.to_numeric(cleaned, errors="coerce")


def load_national_file(path: Path) -> pd.DataFrame:
    data = pd.read_csv(path, dtype=str)
    data.columns = [column.strip().upper() for column in data.columns]
    data = data.rename(columns={"OCC_GROUP": "GROUP", "O_GROUP": "GROUP"})
    data = data.drop(columns=[column for column in HOURLY_COLUMNS if column in data.columns])
    data["YEAR"] = extract_year(path)
    data["SOURCE_FILE"] = str(path.relative_to(PROJECT_ROOT))

    for column in NUMERIC_COLUMNS:
        if column in data.columns:
            data[column] = clean_numeric(data[column])

    data["GROUP"] = data.get("GROUP", "").fillna("").astype(str).str.lower().str.strip()
    return data


raw_panel = pd.concat([load_national_file(path) for path in national_files], ignore_index=True)
print(raw_panel.shape)
raw_panel.head()

(19638, 27)


,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,A_MEAN,MEAN_PRSE,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,YEAR,SOURCE_FILE,AREA,AREA_TITLE,AREA_TYPE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,JOBS_1000,LOC_QUOTIENT,PCT_TOTAL,PRIM_STATE,PCT_RPT
0,00-0000,All Occupations,total,127097160,0.1,44410,0.1,17690,22150,33840,54250,83140,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11-0000,Management Occupations,major,6022860,0.2,105440,0.1,44860,63760,91440,130980,<NA>,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11-1011,Chief Executives,,273500,0.5,173350,0.3,75160,107990,165080,<NA>,<NA>,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11-1021,General and Operations Managers,,1708080,0.3,113100,0.2,47280,65290,94400,142030,<NA>,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11-1031,Legislators,,65710,1.3,38470,1.2,15790,16790,19260,54170,84320,True,2010,converted_csv/oesm10nat/national_M2010_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Use detailed occupations only. Older files mark details with a blank group; newer files use 'detailed'.
detail_groups = {"", "detailed"}
panel = raw_panel[raw_panel["GROUP"].isin(detail_groups)].copy()
panel = panel[panel["OCC_CODE"].notna() & (panel["OCC_CODE"] != "00-0000")]
panel = panel.sort_values(["OCC_CODE", "YEAR"]).reset_index(drop=True)

# Add stable occupation-code features. These help the model generalize across related occupations.
panel["OCC_MAJOR"] = panel["OCC_CODE"].str.slice(0, 2)
panel["OCC_MINOR"] = panel["OCC_CODE"].str.slice(0, 5)
panel["LOG_TOT_EMP"] = np.log1p(panel["TOT_EMP"])
panel["LOG_A_MEAN"] = np.log1p(panel["A_MEAN"])
panel["EMPLOYMENT_SHARE"] = panel["TOT_EMP"] / panel.groupby("YEAR")["TOT_EMP"].transform("sum")

print("Years:", sorted(panel["YEAR"].unique()))
print("Detailed occupation rows:", panel.shape[0])
print("Unique occupations:", panel["OCC_CODE"].nunique())
panel.head()

Years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Detailed occupation rows: 12210
Unique occupations: 984


,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,A_MEAN,MEAN_PRSE,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,YEAR,SOURCE_FILE,AREA,AREA_TITLE,AREA_TYPE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,JOBS_1000,LOC_QUOTIENT,PCT_TOTAL,PRIM_STATE,PCT_RPT,OCC_MAJOR,OCC_MINOR,LOG_TOT_EMP,LOG_A_MEAN,EMPLOYMENT_SHARE
0,11-1011,Chief Executives,,273500,0.5,173350,0.3,75160,107990,165080,<NA>,<NA>,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,11-10,12.519061,12.063074,0.002152
1,11-1011,Chief Executives,,267370,0.5,176550,0.4,75860,109320,166910,<NA>,<NA>,NaN,2011,oesm11nat/national_M2011_dl.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,11-10,12.496392,12.081365,0.002084
2,11-1011,Chief Executives,detailed,255940,0.6,176840,0.3,76220,109940,168140,<NA>,<NA>,NaN,2012,converted_csv/oesm12nat/national_M2012_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,11-10,12.452702,12.083006,0.001964
3,11-1011,Chief Executives,detailed,248760,0.6,178400,0.3,75030,110610,171610,<NA>,<NA>,NaN,2013,converted_csv/oesm13nat/national_M2013_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,11-10,12.424248,12.091789,0.001876
4,11-1011,Chief Executives,detailed,246240,0.8,180700,0.4,72750,110760,173320,<NA>,<NA>,NaN,2014,converted_csv/oesm14nat/national_M2014_dl__nat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,11-10,12.414066,12.104599,0.001822


In [5]:
# Create one-year-ahead targets by occupation.
panel["NEXT_YEAR"] = panel.groupby("OCC_CODE")["YEAR"].shift(-1)
panel["NEXT_TOT_EMP"] = panel.groupby("OCC_CODE")["TOT_EMP"].shift(-1)
panel["NEXT_A_MEAN"] = panel.groupby("OCC_CODE")["A_MEAN"].shift(-1)
panel["TARGET_LOG_TOT_EMP"] = np.log1p(panel["NEXT_TOT_EMP"])
panel["TARGET_LOG_A_MEAN"] = np.log1p(panel["NEXT_A_MEAN"])

model_data = panel[
    panel["NEXT_YEAR"].eq(panel["YEAR"] + 1)
    & panel["TARGET_LOG_TOT_EMP"].notna()
    & panel["TARGET_LOG_A_MEAN"].notna()
].copy()

print(model_data[["YEAR", "NEXT_YEAR"]].drop_duplicates().sort_values(["YEAR", "NEXT_YEAR"]))
print("Model rows:", model_data.shape[0])

    YEAR  NEXT_YEAR
0   2010     2011.0
1   2011     2012.0
2   2012     2013.0
3   2013     2014.0
4   2014     2015.0
5   2015     2016.0
6   2016     2017.0
7   2017     2018.0
8   2018     2019.0
9   2019     2020.0
10  2020     2021.0
11  2021     2022.0
12  2022     2023.0
13  2023     2024.0
Model rows: 11149


In [6]:
FEATURE_COLUMNS = [
    "YEAR",
    "OCC_CODE",
    "OCC_TITLE",
    "OCC_MAJOR",
    "OCC_MINOR",
    "TOT_EMP",
    "EMP_PRSE",
    "A_MEAN",
    "MEAN_PRSE",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90",
    "LOG_TOT_EMP",
    "LOG_A_MEAN",
    "EMPLOYMENT_SHARE",
]
TARGET_COLUMNS = ["TARGET_LOG_TOT_EMP", "TARGET_LOG_A_MEAN"]
TRAIN_TARGET_END_YEAR = 2022
TEST_TARGET_YEARS = [2023, 2024]

available_features = [column for column in FEATURE_COLUMNS if column in model_data.columns]
categorical_features = ["OCC_CODE", "OCC_TITLE", "OCC_MAJOR", "OCC_MINOR"]
numeric_features = [column for column in available_features if column not in categorical_features]

train_data = model_data[model_data["NEXT_YEAR"].le(TRAIN_TARGET_END_YEAR)].copy()
test_data = model_data[model_data["NEXT_YEAR"].isin(TEST_TARGET_YEARS)].copy()

print(f"Training target years: {int(train_data['NEXT_YEAR'].min())}-{int(train_data['NEXT_YEAR'].max())}")
print("Testing target years:", sorted(test_data["NEXT_YEAR"].astype(int).unique()))
print("Train rows:", train_data.shape[0], "Test rows:", test_data.shape[0])

Training target years: 2011-2022
Testing target years: [np.int64(2023), np.int64(2024)]
Train rows: 9498 Test rows: 1651


In [7]:
# Export the cleaned, model-ready dataset so others can train or test their own models.
export_path = PROJECT_ROOT / "annual_wage_model_data_2010_2024.csv"
export_columns = list(
    dict.fromkeys(
        [
            "YEAR",
            "NEXT_YEAR",
            "OCC_CODE",
            "OCC_TITLE",
            "SOURCE_FILE",
            *available_features,
            "NEXT_TOT_EMP",
            "NEXT_A_MEAN",
            *TARGET_COLUMNS,
        ]
    )
)

model_export = model_data[[column for column in export_columns if column in model_data.columns]].copy()
model_export["DATA_SPLIT"] = "unused"
model_export.loc[model_export["NEXT_YEAR"].le(TRAIN_TARGET_END_YEAR), "DATA_SPLIT"] = "train_through_2022"
model_export.loc[model_export["NEXT_YEAR"].isin(TEST_TARGET_YEARS), "DATA_SPLIT"] = "holdout_2023_2024"
model_export = model_export.rename(columns={"NEXT_YEAR": "TARGET_YEAR"})
model_export["TARGET_YEAR"] = model_export["TARGET_YEAR"].astype(int)
model_export.to_csv(export_path, index=False)

print(f"Wrote {model_export.shape[0]} rows and {model_export.shape[1]} columns to {export_path.relative_to(PROJECT_ROOT)}")
model_export.head()

Wrote 11149 rows and 24 columns to annual_wage_model_data_2010_2024.csv


,YEAR,TARGET_YEAR,OCC_CODE,OCC_TITLE,SOURCE_FILE,OCC_MAJOR,OCC_MINOR,TOT_EMP,EMP_PRSE,A_MEAN,MEAN_PRSE,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,LOG_TOT_EMP,LOG_A_MEAN,EMPLOYMENT_SHARE,NEXT_TOT_EMP,NEXT_A_MEAN,TARGET_LOG_TOT_EMP,TARGET_LOG_A_MEAN,DATA_SPLIT
0,2010,2011,11-1011,Chief Executives,converted_csv/oesm10nat/national_M2010_dl__nat...,11,11-10,273500,0.5,173350,0.3,75160,107990,165080,<NA>,<NA>,12.519061,12.063074,0.002152,267370,176550,12.496392,12.081365,train_through_2022
1,2011,2012,11-1011,Chief Executives,oesm11nat/national_M2011_dl.csv,11,11-10,267370,0.5,176550,0.4,75860,109320,166910,<NA>,<NA>,12.496392,12.081365,0.002084,255940,176840,12.452702,12.083006,train_through_2022
2,2012,2013,11-1011,Chief Executives,converted_csv/oesm12nat/national_M2012_dl__nat...,11,11-10,255940,0.6,176840,0.3,76220,109940,168140,<NA>,<NA>,12.452702,12.083006,0.001964,248760,178400,12.424248,12.091789,train_through_2022
3,2013,2014,11-1011,Chief Executives,converted_csv/oesm13nat/national_M2013_dl__nat...,11,11-10,248760,0.6,178400,0.3,75030,110610,171610,<NA>,<NA>,12.424248,12.091789,0.001876,246240,180700,12.414066,12.104599,train_through_2022
4,2014,2015,11-1011,Chief Executives,converted_csv/oesm14nat/national_M2014_dl__nat...,11,11-10,246240,0.8,180700,0.4,72750,110760,173320,<NA>,<NA>,12.414066,12.104599,0.001822,238940,185850,12.383972,12.132701,train_through_2022


In [8]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", min_frequency=2)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "forest",
            RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=1,
            ),
        ),
    ]
)

model.fit(train_data[available_features], train_data[TARGET_COLUMNS])
predicted_log = model.predict(test_data[available_features])
predictions = pd.DataFrame(
    {
        "YEAR": test_data["YEAR"].to_numpy(),
        "TARGET_YEAR": test_data["NEXT_YEAR"].to_numpy().astype(int),
        "OCC_CODE": test_data["OCC_CODE"].to_numpy(),
        "OCC_TITLE": test_data["OCC_TITLE"].to_numpy(),
        "ACTUAL_TOT_EMP": test_data["NEXT_TOT_EMP"].to_numpy(),
        "PREDICTED_TOT_EMP": np.expm1(predicted_log[:, 0]),
        "ACTUAL_A_MEAN": test_data["NEXT_A_MEAN"].to_numpy(),
        "PREDICTED_A_MEAN": np.expm1(predicted_log[:, 1]),
    }
)
predictions.head()

,YEAR,TARGET_YEAR,OCC_CODE,OCC_TITLE,ACTUAL_TOT_EMP,PREDICTED_TOT_EMP,ACTUAL_A_MEAN,PREDICTED_A_MEAN
0,2022,2023,11-1011,Chief Executives,211230,2.065017e+05,258900,198943.388871
1,2023,2024,11-1011,Chief Executives,211850,2.072283e+05,262930,199212.972563
2,2022,2023,11-1021,General and Operations Managers,3507810,2.661437e+06,129330,115804.220381
3,2023,2024,11-1021,General and Operations Managers,3584420,2.660876e+06,133120,117082.601058
4,2022,2023,11-1031,Legislators,32460,4.481688e+04,68140,72305.397644


In [9]:
def regression_report(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    actual = pd.Series(actual, dtype="float64")
    predicted = pd.Series(predicted, dtype="float64")
    wape = actual.sub(predicted).abs().sum() / actual.abs().sum()
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "WAPE": wape,
        "ACCURACY_PERCENT": (1 - wape) * 100,
        "R2": r2_score(actual, predicted),
    }


def make_metrics(data: pd.DataFrame, label: str) -> list[dict[str, float | str | int]]:
    return [
        {"split": label, "target": "TOT_EMP", **regression_report(data["ACTUAL_TOT_EMP"], data["PREDICTED_TOT_EMP"])},
        {"split": label, "target": "A_MEAN", **regression_report(data["ACTUAL_A_MEAN"], data["PREDICTED_A_MEAN"])},
    ]


metrics = pd.DataFrame(make_metrics(predictions, "2023-2024 holdout"))
metrics_by_year = pd.DataFrame(
    row
    for target_year, year_predictions in predictions.groupby("TARGET_YEAR")
    for row in make_metrics(year_predictions, int(target_year))
)

display(metrics)
metrics_by_year

,split,target,MAE,RMSE,WAPE,ACCURACY_PERCENT,R2
0,2023-2024 holdout,TOT_EMP,11332.908464,44896.807704,0.061186,93.881375,0.989703
1,2023-2024 holdout,A_MEAN,4618.634455,16875.525587,0.060154,93.984567,0.886631


,split,target,MAE,RMSE,WAPE,ACCURACY_PERCENT,R2
0,2023,TOT_EMP,10563.390887,40812.921555,0.057440,94.256041,0.991376
1,2023,A_MEAN,4571.227381,16028.702752,0.060604,93.939587,0.895487
2,2024,TOT_EMP,12101.494421,48634.641507,0.064876,93.512437,0.988076
3,2024,A_MEAN,4665.984137,17680.884651,0.059721,94.027938,0.877982


In [10]:
predictions["ABS_EMP_ERROR"] = (predictions["ACTUAL_TOT_EMP"] - predictions["PREDICTED_TOT_EMP"]).abs()
predictions["EMP_PCT_ERROR"] = predictions["ABS_EMP_ERROR"] / predictions["ACTUAL_TOT_EMP"]
predictions["ABS_WAGE_ERROR"] = (predictions["ACTUAL_A_MEAN"] - predictions["PREDICTED_A_MEAN"]).abs()
predictions["WAGE_PCT_ERROR"] = predictions["ABS_WAGE_ERROR"] / predictions["ACTUAL_A_MEAN"]

largest_emp_errors = predictions.sort_values("ABS_EMP_ERROR", ascending=False).head(15)
largest_emp_errors[[
    "OCC_CODE",
    "OCC_TITLE",
    "ACTUAL_TOT_EMP",
    "PREDICTED_TOT_EMP",
    "EMP_PCT_ERROR",
    "ACTUAL_A_MEAN",
    "PREDICTED_A_MEAN",
    "WAGE_PCT_ERROR",
]]

,OCC_CODE,OCC_TITLE,ACTUAL_TOT_EMP,PREDICTED_TOT_EMP,EMP_PCT_ERROR,ACTUAL_A_MEAN,PREDICTED_A_MEAN,WAGE_PCT_ERROR
3,11-1021,General and Operations Managers,3584420,2.660876e+06,0.257655,133120,117082.601058,0.120473
2,11-1021,General and Operations Managers,3507810,2.661437e+06,0.241282,129330,115804.220381,0.104583
742,31-1120,Home Health and Personal Care Aides,3988140,3.467813e+06,0.130469,34990,30668.227702,0.123514
640,29-1141,Registered Nurses,3282010,2.926422e+06,0.108345,98430,83379.366280,0.152907
844,35-3023,Fast Food and Counter Workers,3780930,3.447774e+06,0.088115,31350,30397.496300,0.030383
1632,53-7062,"Laborers and Freight, Stock, and Material Move...",2982530,2.662136e+06,0.107424,41420,39876.011699,0.037276
843,35-3023,Fast Food and Counter Workers,3676580,3.363612e+06,0.085125,30110,29426.866413,0.022688
954,41-2031,Retail Salespersons,3800250,3.533288e+06,0.070249,37150,31072.457377,0.163595
1017,43-4051,Customer Service Representatives,2858710,2.605482e+06,0.088581,43520,40108.929271,0.078379
639,29-1141,Registered Nurses,3175390,2.925018e+06,0.078848,94480,82317.657748,0.128729


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(predictions["ACTUAL_TOT_EMP"], predictions["PREDICTED_TOT_EMP"], alpha=0.55)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Employment: Actual vs Predicted")
axes[0].set_xlabel("Actual next-year employment")
axes[0].set_ylabel("Predicted next-year employment")

axes[1].scatter(predictions["ACTUAL_A_MEAN"], predictions["PREDICTED_A_MEAN"], alpha=0.55)
axes[1].set_title("Annual Mean Wage: Actual vs Predicted")
axes[1].set_xlabel("Actual next-year annual mean wage")
axes[1].set_ylabel("Predicted next-year annual mean wage")

plt.tight_layout()

In [12]:
# Train on all known one-year transitions, then forecast one year beyond the latest downloaded year.
final_model = model
final_model.fit(model_data[available_features], model_data[TARGET_COLUMNS])

latest_year = int(panel["YEAR"].max())
latest_features = panel[panel["YEAR"].eq(latest_year)].copy()
latest_predictions_log = final_model.predict(latest_features[available_features])

next_year_forecast = pd.DataFrame(
    {
        "FORECAST_FROM_YEAR": latest_year,
        "TARGET_YEAR": latest_year + 1,
        "OCC_CODE": latest_features["OCC_CODE"].to_numpy(),
        "OCC_TITLE": latest_features["OCC_TITLE"].to_numpy(),
        "CURRENT_TOT_EMP": latest_features["TOT_EMP"].to_numpy(),
        "FORECAST_TOT_EMP": np.expm1(latest_predictions_log[:, 0]),
        "CURRENT_A_MEAN": latest_features["A_MEAN"].to_numpy(),
        "FORECAST_A_MEAN": np.expm1(latest_predictions_log[:, 1]),
    }
)
next_year_forecast["FORECAST_EMP_CHANGE"] = next_year_forecast["FORECAST_TOT_EMP"] - next_year_forecast["CURRENT_TOT_EMP"]
next_year_forecast["FORECAST_EMP_GROWTH"] = next_year_forecast["FORECAST_EMP_CHANGE"] / next_year_forecast["CURRENT_TOT_EMP"]
next_year_forecast.sort_values("FORECAST_EMP_CHANGE", ascending=False).head(20)

,FORECAST_FROM_YEAR,TARGET_YEAR,OCC_CODE,OCC_TITLE,CURRENT_TOT_EMP,FORECAST_TOT_EMP,CURRENT_A_MEAN,FORECAST_A_MEAN,FORECAST_EMP_CHANGE,FORECAST_EMP_GROWTH
790,2024,2025,53-3032,Heavy and Tractor-Trailer Truck Drivers,2070480,2.251059e+06,58400.0,62725.710198,180578.906647,0.087216
49,2024,2025,13-1111,Management Analysts,893900,1.054192e+06,114710.0,107532.797849,160292.397424,0.179318
54,2024,2025,13-1161,Market Research Analysts and Marketing Special...,861140,9.225096e+05,86480.0,90375.729080,61369.554539,0.071265
44,2024,2025,13-1071,Human Resources Specialists,917460,9.623167e+05,79730.0,83103.103467,44856.744292,0.048892
487,2024,2025,41-3091,"Sales Representatives of Services, Except Adve...",1189330,1.231124e+06,81260.0,83438.165879,41793.817980,0.035141
5,2024,2025,11-2022,Sales Managers,603710,6.440487e+05,160930.0,164103.998374,40338.733505,0.066818
8,2024,2025,11-3012,Administrative Services Managers,254140,2.834104e+05,126030.0,131453.507202,29270.372778,0.115174
55,2024,2025,13-1199,"Business Operations Specialists, All Other",1128200,1.156388e+06,92380.0,92272.696161,28188.025750,0.024985
84,2024,2025,15-1299,"Computer Occupations, All Other",439380,4.659143e+05,116700.0,118944.667633,26534.258711,0.060390
99,2024,2025,17-2051,Civil Engineers,355410,3.754396e+05,107050.0,110056.745275,20029.640513,0.056356


## Interpretation and Next Steps

The model predicts each detailed occupation's next-year employment and annual mean wage. It is a regression model, so the accuracy score is based on prediction error rather than classification correctness. The headline `ACCURACY_PERCENT` is `1 - WAPE`, meaning it measures how close the model's total absolute prediction error is to the total actual value on the 2023-2024 holdout set.

The holdout results should be treated as a backtest of one-year-ahead predictions. The final forecast cell then retrains on all known one-year transitions through 2024 and projects one year beyond the latest downloaded data.

Recommended final project workflow:

1. Compare Random Forest against simple baselines such as last-year carry-forward and linear trend.
2. Add prediction intervals or scenario ranges before presenting forecasts as decisions.
3. Report uncertainty clearly, especially for occupations affected by SOC classification changes or BLS suppression symbols.
4. Only try RNN/LSTM if you add much more frequent data, such as monthly job postings or quarterly labor-market indicators. Annual BLS files alone are too short for a reliable deep learning sequence model.